### Semantic Chunking
Semantic Chunking is a document splitter that uses embedding similarity between sentences to decide chunk boundaries. It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.
Steps to be followed.
<ol>
<li>Document Segmentation</li>
<li>Sentence Embedding</li>
<li>Semantic Similarity Check</li>
<li>Merging of Sentences based on Cosine Similarity</li>
<li>Form the chunks based on similarity</li>
</ol>

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [27]:
# Initialize the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample text
text="""LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is lcoated in Paris.
France is a popular tourist destination."""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Document Segmentation

In [28]:
text

'LangChain is a framework for building applications with LLMs.\nLangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is lcoated in Paris.\nFrance is a popular tourist destination.'

In [29]:
# Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]
sentences

['LangChain is a framework for building applications with LLMs.',
 'LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers.',
 'The Eiffel Tower is lcoated in Paris.',
 'France is a popular tourist destination.']

### Sentence Embedding

In [30]:
# Embed each sentence
embeddings=model.encode(sentences)
embeddings

array([[-0.02109225, -0.04472173,  0.01087079, ..., -0.01217804,
         0.08605656,  0.02890728],
       [-0.03418021, -0.1021043 ,  0.00366989, ..., -0.01398786,
         0.04454357,  0.0055136 ],
       [-0.02442168, -0.05424954, -0.13623358, ...,  0.0365635 ,
         0.07216298, -0.03104779],
       [ 0.07061954,  0.05062735,  0.03053246, ..., -0.00474113,
         0.08106465,  0.06758431],
       [ 0.1040301 , -0.03097698,  0.02524884, ...,  0.07805592,
         0.01353771, -0.02684899]], dtype=float32)

### Semantic Similarity Check

In [31]:
# Initiliaze parameters
threshold = 0.7 # Control chunk tightness
chunks = []
current_chunk = [sentences[0]] # Start with the first sentence
current_chunk

['LangChain is a framework for building applications with LLMs.']

### Merging of Sentences based on Cosine Similarity

In [32]:
# Semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]  

# Add the last chunk
if current_chunk:
    chunks.append(" ".join(current_chunk))  

#Output the chunks
print("Semantic Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Semantic Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 2: You can create chains, agents, memory, and retrievers.
Chunk 3: The Eiffel Tower is lcoated in Paris.
Chunk 4: France is a popular tourist destination.
